Starting kit application with the following args:  ['/isaac-sim/exts/isaacsim.simulation_app/isaacsim/simulation_app/simulation_app.py', '/isaac-sim/apps/isaacsim.exp.action_and_event_data_generation.base.kit', '--/app/tokens/exe-path=/isaac-sim/kit', '--/persistent/app/viewport/displayOptions=3094', '--/rtx/materialDb/syncLoads=True', '--/rtx/hydra/materialSyncLoads=True', '--/omni.kit.plugin/syncUsdLoads=True', '--/app/renderer/resolution/width=1280', '--/app/renderer/resolution/height=720', '--/app/window/width=1440', '--/app/window/height=900', '--/renderer/multiGpu/enabled=True', '--/app/fastShutdown=True', '--/app/installSignalHandlers=0', '--ext-folder', '/isaac-sim/exts', '--ext-folder', '/isaac-sim/apps', '--/physics/cudaDevice=0', '--/plugins/carb.tasking.plugin/threadCount=32', '--/plugins/omni.tbb.globalcontrol/maxThreadCount=32', '--portable', '--no-window', '--/app/window/hideUi=1', '--allow-root']
Passing the following args to the base kit application:  ['--f=/root/.loca

AttributeError: '_UnixSelectorEventLoop' object has no attribute '_old_agen_hooks'

2026-04-19T01:31:37Z [17,923ms] [Warning] [omni.fabric.plugin] Warning: attribute viewportHandle not found for bucket id 9



2026-04-19T01:31:37Z [17,945ms] [Error] [asyncio] Exception in callback <TaskStepMethWrapper object at 0x7cc1a8830a00>()
handle: <Handle <TaskStepMethWrapper object at 0x7cc1a8830a00>()>
Traceback (most recent call last):
  File "/isaac-sim/kit/python/lib/python3.11/asyncio/events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: Cannot enter into task <Task pending name='Task-58' coro=<MainWindow._dock_windows() running at /isaac-sim/extscache/omni.kit.mainwindow-1.0.3+69cbf6ad/omni/kit/mainwindow/scripts/main_window.py:71>> while another task <Task pending name='Task-57' coro=<Kernel.shell_main() running at /root/.local/lib/python3.11/site-packages/ipykernel/kernelbase.py:621> cb=[Task.task_wakeup()]> is being executed.
2026-04-19T01:31:37Z [17,946ms] [Error] [asyncio] Exception in callback <TaskStepMethWrapper object at 0x7cc19b6b9a50>()
handle: <Handle <TaskStepMethWrapper object at 0x7cc19b6b9a50>()>
Traceback (most recent call last):
  File "/

[18.010s] app ready


2026-04-19T01:31:38Z [18,328ms] [Error] [omni.kit.app._impl] [py stderr]: /isaac-sim/extscache/omni.kit.test-2.0.1+69cbf6ad.lx64.r.cp311/omni/kit/test/__init__.py:95: RuntimeWarning: coroutine 'UsdExtension.__init_stage_event' was never awaited
  def on_app_ready(_):
2026-04-19T01:31:38Z [18,328ms] [Error] [asyncio] Task was destroyed but it is pending!
task: <Task pending name='Task-59' coro=<UsdExtension.__init_stage_event() running at /isaac-sim/extscache/omni.usd-1.13.10+69cbf6ad.lx64.r.cp311/omni/usd/_impl/__init__.py:29>>
2026-04-19T01:31:38Z [18,328ms] [Error] [omni.kit.app._impl] [py stderr]: /isaac-sim/extscache/omni.kit.test-2.0.1+69cbf6ad.lx64.r.cp311/omni/kit/test/__init__.py:95: RuntimeWarning: coroutine 'DetailView._build_detail_frames.<locals>.build_frame_async' was never awaited
  def on_app_ready(_):
2026-04-19T01:31:38Z [18,328ms] [Error] [asyncio] Task was destroyed but it is pending!
task: <Task pending name='Task-68' coro=<DetailView._build_detail_frames.<locals>.b

[20.528s] Simulation App Startup Complete
Stage default prim: /dingo
Root layer: /workspace/FLUX/assets/robots/dingo.usd


In [ ]:
# ── 1. Launch IsaacSim ──────────────────────────────────────────────────────
from isaacsim import SimulationApp
import os
CUSTOM_APP_PATH = os.path.join(
    os.environ["EXP_PATH"],
    "isaacsim.exp.action_and_event_data_generation.base.kit"
)

simulation_app = SimulationApp(
    launch_config={
        "renderer": "RayTracedLighting",
        "headless": True,
        "enable_cameras": True,
    },
    experience=CUSTOM_APP_PATH,
)
# cell 1: 打开 USD
from pxr import Usd, UsdGeom, Sdf

USD_PATH = "/workspace/FLUX/assets/robots/dingo.usd"

stage = Usd.Stage.Open(USD_PATH)
print(f"Stage default prim: {stage.GetDefaultPrim().GetPath() if stage.GetDefaultPrim() else None}")
print(f"Root layer: {stage.GetRootLayer().identifier}")
# cell 2: 确认 GroundPlane 存在 + 查看它的位置
# 因为这是编辑 dingo.usd 本身(不是场景 reference),路径应该是 /dingo/GroundPlane
# 或 /GroundPlane,取决于 USD 里 default prim 名字。先列出来看看。

for prim in stage.Traverse():
    path = str(prim.GetPath())
    if "GroundPlane" in path or "ground" in path.lower():
        print(f"  Found: {path}  type={prim.GetTypeName()}")
# cell 3: 删除 GroundPlane
# 根据上一个 cell 的打印结果,填入正确的完整路径

# 例子:如果上一步打印的是 "/dingo/GroundPlane",用这个:
GROUND_PATH = "/dingo/GroundPlane"   # ← 根据 cell 2 实际结果修改

# 验证路径有效
prim = stage.GetPrimAtPath(GROUND_PATH)
if not prim.IsValid():
    print(f"ERROR: {GROUND_PATH} not valid, check cell 2 output")
else:
    print(f"Removing: {GROUND_PATH}")
    stage.RemovePrim(GROUND_PATH)
    print("Removed.")
# cell 4: 保存(就地覆盖 dingo.usd)
stage.GetRootLayer().Save()
print(f"Saved to {USD_PATH}")